In [92]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# On essaie d'utiliser __file__, sinon on prend le dossier actuel
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

ROOT = BASE_DIR.parent
DATA_DIR = ROOT / "data"

%ls ../data/processed/data/commune

print(DATA_DIR)

commune_clusters.parquet               raw_comm_score_apl_niv0.parquet
commune_niv1_score_irdes.csv           score_sante_territoires_final.csv
commune_niv1_score_irdes.parquet       score_sante_territoires_final.parquet
raw_comm_score_apl_niv0.csv
/Users/jean-jacques/code/jjchabutDataCRM/sante-territoires/data


In [93]:
GPKG = DATA_DIR / 'raw/territoires/ADE-COG_4-0_GPKG_LAMB93_FXX-ED2025-01-01.gpkg'

# Communes
gdf_communes = gpd.read_file(GPKG, layer='commune')

gdf_communes.columns

Index(['cleabs', 'nom_officiel', 'nom_officiel_en_majuscules', 'statut',
       'code_insee', 'population', 'date_du_recensement',
       'organisme_recenseur', 'code_insee_du_canton',
       'code_insee_de_l_arrondissement', 'code_insee_du_departement',
       'code_insee_de_la_region', 'code_siren', 'codes_siren_des_epci',
       'code_postal', 'superficie_cadastrale', 'geometry'],
      dtype='object')

In [123]:
# Ce fichier contient les communes + les arrondiments municipaux (Lyon, Marseille, Paris)
file_path = '../data/processed/ref/ref_comm_plus_arrond.parquet'


df_ref_comm =pd.read_parquet(file_path)

In [124]:
df_ref_comm.columns

Index(['code_insee', 'nom_commune', 'population', 'superficie',
       'code_insee_du_departement', 'codes_siren_des_epci',
       'code_insee_de_la_region'],
      dtype='object')

In [113]:
df_ref_comm.shape

(34791, 6)

In [125]:
df_ref_comm.isna().sum()

code_insee                    0
nom_commune                   0
population                    0
superficie                   45
code_insee_du_departement     0
codes_siren_des_epci         45
code_insee_de_la_region       0
dtype: int64

In [126]:
df_75 = df_ref_comm[df_ref_comm['code_insee'].str.startswith('75')]

df_75.head()

,code_insee,nom_commune,population,superficie,code_insee_du_departement,codes_siren_des_epci,code_insee_de_la_region
29194,75056,Paris,2113705,10540.0,75,200054781,11
34747,75119,Paris 19e Arrondissement,178371,NaN,75,None,11
34748,75118,Paris 18e Arrondissement,185825,NaN,75,None,11
34749,75115,Paris 15e Arrondissement,228754,NaN,75,None,11
34750,75114,Paris 14e Arrondissement,137581,NaN,75,None,11


In [127]:
file_apl = DATA_DIR / 'processed/data/commune' / 'raw_comm_score_apl_niv0.parquet'

df_apl = pd.read_parquet(file_apl)

df_apl.head()

,code_insee_comm,nom_commune,apl_medecins,apl_65ans,apl_62ans,apl_60ans,pop_standard_2021,pop_totale_2021,apl_dentistes,apl_infirmiers,...,delta_apl_infirmiers,delta_apl_kines,delta_apl_sagesfemmes,apl_medecins_std,apl_dentistes_std,apl_infirmiers_std,apl_kines_std,apl_sagesfemmes_std,score_apl,quintile_apl_nat
0,01001,L'Abergement-Clémenciat,1.942,1.881,1.623,1.455,838.154,832,35.270,122.416,...,14.903,-2.978,12.955,-0.772546,0.026716,0.044420,-0.610034,1.186782,-0.317946,Q2 (faible)
1,01002,L'Abergement-de-Varey,2.376,1.767,1.503,1.333,255.723,267,36.427,109.849,...,-95.639,-71.597,-14.449,-0.442053,0.076964,-0.140549,-0.510025,-0.214993,-0.291066,Q2 (faible)
2,01004,Ambérieu-en-Bugey,3.083,2.431,2.136,1.855,14575.887,14854,59.621,201.121,...,60.097,24.294,-1.842,0.096332,1.084259,1.202849,1.084761,1.284043,0.778222,Q5 (très bon)
3,01005,Ambérieux-en-Dombes,3.706,3.648,3.015,2.998,1852.496,1897,50.539,131.876,...,85.359,92.246,17.529,0.570751,0.689836,0.183658,0.774416,1.587885,0.583430,Q5 (très bon)
4,01006,Ambléon,0.889,0.775,0.648,0.570,121.314,113,10.929,44.227,...,-153.632,-89.582,-15.397,-1.574412,-1.030392,-1.106414,-1.130735,-0.453044,-1.231006,Q1 (très faible)


In [118]:
# Tester code insee 75119 (Paris 19e arrond)

df_apl_75019 = df_apl[df_apl['code_insee_comm'] == '75119']

df_apl_75019.head()

,code_insee_comm,nom_commune,apl_medecins,apl_65ans,apl_62ans,apl_60ans,pop_standard_2021,pop_totale_2021,apl_dentistes,apl_infirmiers,...,delta_apl_infirmiers,delta_apl_kines,delta_apl_sagesfemmes,apl_medecins_std,apl_dentistes_std,apl_infirmiers_std,apl_kines_std,apl_sagesfemmes_std,score_apl,quintile_apl_nat
29279,75119,Paris 19e Arrondissement,5.289,4.435,4.05,3.828,174095.775,181616,113.42,96.954,...,11.247,2.128,1.527,1.776215,3.420703,-0.330345,0.954751,0.291046,1.257697,Q5 (très bon)


In [119]:
print(f'Apl : {df_apl.shape}')
print(f'Comm : {df_ref_comm.shape}')

Apl : (34954, 29)
Comm : (34791, 6)


In [128]:
# Fusion avec indicateur
df_merge = pd.merge(
    df_apl, 
    df_ref_comm, 
    left_on="code_insee_comm", 
    right_on="code_insee",
    how="outer", 
    indicator= True
)


In [129]:
df_merge.shape

# (34774, 35)

(34971, 37)

In [130]:
df_merge.columns

Index(['code_insee_comm', 'nom_commune_x', 'apl_medecins', 'apl_65ans',
       'apl_62ans', 'apl_60ans', 'pop_standard_2021', 'pop_totale_2021',
       'apl_dentistes', 'apl_infirmiers', 'apl_kines', 'apl_sagesfemmes',
       'apl_medecins_2022', 'apl_dentistes_2022', 'apl_infirmiers_2022',
       'apl_kines_2022', 'apl_sagesfemmes_2022', 'delta_apl_medecins',
       'delta_apl_dentistes', 'delta_apl_infirmiers', 'delta_apl_kines',
       'delta_apl_sagesfemmes', 'apl_medecins_std', 'apl_dentistes_std',
       'apl_infirmiers_std', 'apl_kines_std', 'apl_sagesfemmes_std',
       'score_apl', 'quintile_apl_nat', 'code_insee', 'nom_commune_y',
       'population', 'superficie', 'code_insee_du_departement',
       'codes_siren_des_epci', 'code_insee_de_la_region', '_merge'],
      dtype='object')

In [122]:
# 1. On identifie les codes qui sont dans APL mais PAS dans le Référentiel
manquants_dans_ref = df_apl[~df_apl['code_insee_comm'].isin(df_ref_comm['code_insee'])]

# 2. On identifie les codes qui sont dans le Référentiel mais PAS dans APL
manquants_dans_apl = df_ref_comm[~df_ref_comm['code_insee'].isin(df_apl['code_insee_comm'])]

print(f"Lignes APL orphelines : {len(manquants_dans_ref)}")
print(manquants_dans_ref[['code_insee_comm', 'nom_commune']].head(10))

print(f"\nLignes Référentiel sans APL : {len(manquants_dans_apl)}")
print(manquants_dans_apl[['code_insee', 'nom_commune']].head(20))

Lignes APL orphelines : 180
     code_insee_comm               nom_commune
277            01330                   Ruffieu
690            02311                    Filain
2623           08300              Le Mont-Dieu
3088           09287                  Senconac
4074           12076       Conques-en-Rouergue
4428           14011                Aurseulles
4636           14300                   Gerrots
4851           14623  Saint-Martin-de-Fontenay
5326           16147       Gardes-le-Pontaroux
5396           16226        Montignac-Charente

Lignes Référentiel sans APL : 17
      code_insee               nom_commune
4201       12218       Conques-en-Rouergue
4338       13055                 Marseille
4805       14581                Aurseulles
4958       15031                    Celles
4962       15035              Chalinargues
4972       15047                 Chavagnac
5087       15171          Sainte-Anastasie
17537      49126              Orée d'Anjou
19997      55039     Beaumont-en-V

In [ ]:
df_merge.drop(columns=['_merge','code_insee','nom_commune_y'], inplace=True)

,code_insee_comm,nom_commune,apl_medecins,apl_65ans,apl_62ans,apl_60ans,pop_standard_2021,pop_totale_2021,apl_dentistes,apl_infirmiers,...,apl_infirmiers_std,apl_kines_std,apl_sagesfemmes_std,score_apl,quintile_apl_nat,population,superficie,code_insee_du_departement,codes_siren_des_epci,code_insee_de_la_region
0,01001,L'Abergement-Clémenciat,1.942,1.881,1.623,1.455,838.154,832.0,35.270,122.416,...,0.044420,-0.610034,1.186782,-0.317946,Q2 (faible),859.0,1590.0,01,200069193,84
1,01002,L'Abergement-de-Varey,2.376,1.767,1.503,1.333,255.723,267.0,36.427,109.849,...,-0.140549,-0.510025,-0.214993,-0.291066,Q2 (faible),273.0,920.0,01,240100883,84
2,01004,Ambérieu-en-Bugey,3.083,2.431,2.136,1.855,14575.887,14854.0,59.621,201.121,...,1.202849,1.084761,1.284043,0.778222,Q5 (très bon),15554.0,2460.0,01,240100883,84
3,01005,Ambérieux-en-Dombes,3.706,3.648,3.015,2.998,1852.496,1897.0,50.539,131.876,...,0.183658,0.774416,1.587885,0.583430,Q5 (très bon),1917.0,1590.0,01,200042497,84
4,01006,Ambléon,0.889,0.775,0.648,0.570,121.314,113.0,10.929,44.227,...,-1.106414,-1.130735,-0.453044,-1.231006,Q1 (très faible),114.0,590.0,01,200040350,84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34966,97420,Sainte-Suzanne,3.764,3.235,2.640,2.360,22314.190,24293.0,31.136,429.810,...,4.568834,1.703016,3.011700,1.825695,Q5 (très bon),NaN,NaN,NaN,NaN,NaN
34967,97421,Salazie,3.354,2.465,1.576,1.576,6748.400,7243.0,0.000,453.686,...,4.920255,-0.573059,-1.756463,0.907820,Q5 (très bon),NaN,NaN,NaN,NaN,NaN
34968,97422,Le Tampon,5.475,4.497,4.090,3.643,76427.591,81943.0,72.321,492.939,...,5.498005,3.236913,1.878459,3.032427,Q5 (très bon),NaN,NaN,NaN,NaN,NaN
34969,97423,Les Trois-Bassins,5.044,5.044,5.044,5.044,6571.416,6899.0,19.298,321.852,...,2.979842,6.317974,2.818386,2.605810,Q5 (très bon),NaN,NaN,NaN,NaN,NaN


In [140]:
df_merge.rename(columns={'nom_commune_x':'nom_commune'}, inplace=True)

In [141]:
df_merge.shape

(34971, 34)

In [142]:
df_merge.columns

Index(['code_insee_comm', 'nom_commune', 'apl_medecins', 'apl_65ans',
       'apl_62ans', 'apl_60ans', 'pop_standard_2021', 'pop_totale_2021',
       'apl_dentistes', 'apl_infirmiers', 'apl_kines', 'apl_sagesfemmes',
       'apl_medecins_2022', 'apl_dentistes_2022', 'apl_infirmiers_2022',
       'apl_kines_2022', 'apl_sagesfemmes_2022', 'delta_apl_medecins',
       'delta_apl_dentistes', 'delta_apl_infirmiers', 'delta_apl_kines',
       'delta_apl_sagesfemmes', 'apl_medecins_std', 'apl_dentistes_std',
       'apl_infirmiers_std', 'apl_kines_std', 'apl_sagesfemmes_std',
       'score_apl', 'quintile_apl_nat', 'population', 'superficie',
       'code_insee_du_departement', 'codes_siren_des_epci',
       'code_insee_de_la_region'],
      dtype='object')

### Les données APL (2023) ont été harmonisées avec le référentiel communal du COG 2025 afin d’assurer la cohérence géographique avec les limites administratives actuelles.

In [144]:
file_path = '../data/processed/data/commune/raw_comm_score_apl_niv1.parquet'
df_merge.to_parquet(file_path)